In [ ]:
import os 
import getpass
import pydantic
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig
from chromadb.config import Settings
from crewai.tools import tool

llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")

In [ ]:
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict


class ScanPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    summary: str = Field(description="The key findings or insights about this topic")
    benefits: List[Dict[str, str]] = Field(
        description="The key benifit and details for topic",
        default_factory=list
    )
    references: List[str] = Field(
        description="the key reference design diagram url for discussed topic",
        default_factory=list
    )

In [ ]:
from com.example.agentic.tools.config import _embedder_config_ollama,_embedder_config_openai

# Agent city expert
scan_doc_agent = Agent(
    role="As a software architect with expertise in {topic}",
    goal="Design robust, scalable system architectures that balance performance, maintainability, and cost-effectiveness",
    backstory="With 20+ years of experience building {topic} systems at scale, you've developed a pragmatic approach"
              "to software architecture. You've seen both successful and failed systems and have learned valuable lessons from each." 
              "You balance theoretical best practices with practical constraints and always consider the maintenance and operational" 
              "aspects of your designs."
    tools=[],
    verbose=True,
    knowledge_sources=[text_kw_source],
    embedder=_embedder_config_openai,
    llm=llm,
    allow_delegation=False,
)


# Define Tasks
scan_doc_task = Task(
    description='Extract key Benefits for {topic} and provide a summarized report.',
    expected_output='Output should include topic benifit, summary. output should include references as list',
    agent=scan_doc_agent,
    output_json=ScanPoint,
    output_file="outputs/L006/scan_doc-summary_{run_id}.json"
)


# Review the context you got and expands each topic into full section for a report about {topic}
# Make sure you find top 10 interesting and relevant design url about {topic} and return list of url
# Define Tasks
format_doc_task = Task(
    description='Extract key Benefits for {topic} and provide a summarized report.',
    expected_output='Output should include topic benifit, summary. output should include references as list',
    agent=scan_doc_agent,
    output_json=ScanPoint,
    output_file="outputs/L006/format_doc-summary_{run_id}.json" 
)

In [ ]:

# Assemble the Crew
from crewai import Crew, Process
from datetime import datetime

_exe_date = datetime.now().strftime('%Y-%m-%d')
_exe_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Crew triggered on {_exe_date} with execution id {_exe_id}")

crew = Crew(
    agents=[scan_doc_agent],
    tasks=[scan_doc_task],
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    verbose=True
)

inputs = {
    'topic': 'Microservice Design',
    'date': _exe_date,
    'run_id': _exe_id
}

# Execute the Tasks
result = crew.kickoff(inputs=inputs)
print(result)

In [ ]:
from crewai_tools import RagTool
from crewai_tools.tools.rag import RagToolConfig, VectorDbConfig, ProviderSpec
from crewai.rag.embeddings.providers.openai.types import OpenAIProviderSpec
# Create a RAG tool with custom configuration
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(path="/home/brijeshdhaker/IdeaProjects/crewai_design_document/vactor_store/chroma")
vectordb: VectorDbConfig = {
    "provider": "chromadb",
    "config": {
        "collection_name": "sandbox_documents",
        "persist_directory":"/home/brijeshdhaker/IdeaProjects/crewai_design_document/vactor_store/chroma",
        "allow_reset":True, 
        "is_persistent":True
    }
}

embedding_model: OpenAIProviderSpec = {
    "provider": "openai",
    "config":{
        "model": "nomic-embed-text",
        "api_key":"",
        "platform_url":"http://localhost:11434/v1"
    }
}

config: RagToolConfig = {
    "vectordb": vectordb,
    "embedding_model": embedding_model,
}

rag_tool = RagTool(config=config, summarize=True)

r = rag_tool.run("JPMORGAN")
r

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
import requests
from datetime import datetime
from typing import Dict, Any
from pydantic import BaseModel, Field
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
import json
import chromadb
from chromadb.config import Settings
from crewai.utilities.paths import db_storage_path
import os
from com.example.agentic.tools.config import _embedder_config_ollama

# Create custom storage with specific embedder
custom_storage = KnowledgeStorage(
    embedder=_embedder_config_ollama,
    collection_name="designs"
)


class DesignKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches data from Space News API."""
    limit: int = Field(default=10, description="Number of document to fetch")

    def load_content(self) -> Dict[Any, str]:
        """Fetch and format space news articles."""
        try:
            # Load documents from external source
            json_articles = self.load_json()
            croma_articles = self.load_from_vactor_store("JPMORGAN")

            formatted_document = self.validate_content(json_articles)
            return {self.collection_name: formatted_document}
        except Exception as e:
            raise ValueError(f"Failed to fetch space news: {str(e)}")

    #
    def load_json(self) :
        articles = []
        try:
            work_dir = os.getenv("WORK_DIR")
            with open(f"{work_dir}/docs/json/articals.json", 'r') as file:
                data = json.load(file)
                articles = data.get('results', [])
        except FileNotFoundError:
            print("Error: The file was not found.")
        except json.JSONDecodeError:
            print("Error: Failed to decode JSON (invalid syntax).")
        
        return articles    

        #
    def load_from_vactor_store(self, query: str, top_k: int = 5, score_threshold: float = 0.0) :
        articles = []
        try:

            # Connect to CrewAI's knowledge ChromaDB
            #knowledge_path = os.path.join("/home/brijeshdhaker/IdeaProjects/crewai_design_document/", "knowledge")
            knowledge_path = "/home/brijeshdhaker/IdeaProjects/crewai_design_document/vactor_store/chroma"
            work_dir = os.getenv("WORK_DIR")
            
            if os.path.exists(knowledge_path):
                vactorstore = chromadb.PersistentClient(path=knowledge_path)
                collections = vactorstore.list_collections()
                
                print("Knowledge Collections:")
                for collection in collections:
                    print(f"  - {collection.name}: {collection.count()} documents")
                    
                    # Sample a few documents to verify content
                    if collection.count() > 0:
                        sample = collection.peek(limit=2)
                        print(f"Sample content: {sample['documents'][0]}...")
                    else:
                        print("No knowledge storage found")

            

            print(f"Retrieving documents for query: '{query}'")
            print(f"Top K: {top_k}, Score threshold: {score_threshold}")
            # Generate query embedding
            embedding_manager=EmbeddingManager()
            query_embedding = embedding_manager.embed_query([query])
            
            # Search in vector store
            try:
                collection = vactorstore.get_or_create_collection(
                    name="sandbox_documents",
                    metadata={"description": "Document Embeddings for RAG"}
                )
                print(f"Vector store initialized. Collection: {collection}")
                print(f"Existing documents in collection: {collection.count()}")

                results = collection.query(
                    query_embeddings=query_embedding,
                    n_results=top_k
                )
                
                # Process results
                retrieved_docs = []
                
                if results['documents'] and results['documents'][0]:
                    documents = results['documents'][0]
                    metadatas = results['metadatas'][0]
                    distances = results['distances'][0]
                    ids = results['ids'][0]
                    
                    for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                        # Convert distance to similarity score (ChromaDB uses cosine distance)
                        similarity_score = 1 - distance
                        
                        if similarity_score >= score_threshold:
                            retrieved_docs.append({
                                'id': doc_id,
                                'content': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank': i + 1
                            })
                    
                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                else:
                    print("No documents found")
                
                return retrieved_docs
            
            except Exception as e:
                print(f"Error during retrieval: {e}")
                return []
        
        #
        except FileNotFoundError:
            print("Error: The file was not found.")
        except json.JSONDecodeError:
            print("Error: Failed to decode JSON (invalid syntax).")
        
        return articles  
    
    #    
    def validate_content(self, articles: list) -> str:
        """Format articles into readable text."""
        formatted = "Space News Articles:\n\n"
        for article in articles:
            formatted += f"""
                Title: {article['title']}
                Published: {article['published_at']}
                Summary: {article['summary']}
                News Site: {article['news_site']}
                URL: {article['url']}
                -------------------"""
        return formatted
    
    #
    def add(self) -> None:
        """Process and store the articles."""
        # if len(documents) == 0:
        #     raise ValueError("Number of documents must be greater then 0")
        #     print(f"Adding {len(self.documents)} documents to knowledge store...")
        # self.documents = documents
        content = self.load_content()
        for _, text in content.items():
            chunks = self._chunk_text(text)
            #print(self.get_embeddings)
            self.chunks.extend(chunks)

    #
    def aadd(self) -> None:
        pass

# Create knowledge source
recent_news = DesignKnowledgeSource(
    storage=custom_storage,
    limit=10,
)

recent_news.load_content()



In [ ]:
# Create specialized agent
space_analyst = Agent(
    role="Space News Analyst",
    goal="Answer questions about space news accurately and comprehensively",
    backstory="""You are a space industry analyst with expertise in space exploration,
    satellite technology, and space industry trends. You excel at answering questions
    about space news and providing detailed, accurate information.""",
    knowledge_sources=[recent_news],
    llm= LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")
)

# Create task that handles user questions
analysis_task = Task(
    description="Answer this question about space news: {user_question}",
    expected_output="A detailed answer based on the recent space news articles",
    agent=space_analyst
)

from com.example.agentic.tools.config import _embedder_config_ollama,_embedder_config_openai
# Create and run the crew
crew = Crew(
    agents=[space_analyst],
    tasks=[analysis_task],
    verbose=True,
    process=Process.sequential,
    embedder=_embedder_config_ollama
)

# Example usage
result = crew.kickoff(inputs={"user_question": "What are the latest developments in space exploration?"})

In [ ]:
from github import Github, Auth
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

github = GitHubAPIWrapper()
toolkit = GitHubToolkit.from_github_api_wrapper(github)

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GIT_HUB_TOKEN")

# Use the new Auth class to handle your token
auth = Auth.Token(_git_hub_token)

# Authenticate using a Personal Access Token
github = Github(auth=auth)

# Get a specific repository
repo = github.get_repo(_repo)

print(repo.stargazers_count)

1


In [1]:
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GIT_HUB_TOKEN")

github = GitHubAPIWrapper(
    active_branch="main", 
    github_repository="brijeshdhaker/docker-hadoop-cluster",
    
)
# The wrapper automatically looks for GITHUB_PERSONAL_ACCESS_TOKEN if not passed
toolkit = GitHubToolkit.from_github_api_wrapper(github)
tools = toolkit.get_tools()
for tool in tools:
    print(tool.name)

ValidationError: 1 validation error for GitHubAPIWrapper
  Value error, Did not find github_app_id, please add an environment variable `GITHUB_APP_ID` which contains it, or pass `github_app_id` as a named parameter. [type=value_error, input_value={'active_branch': 'main',.../docker-hadoop-cluster'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

In [2]:
from langchain_community.document_loaders import GithubFileLoader

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

loader = GithubFileLoader(
    repo=_repo,  # the repo name
    branch="main",  # the branch name
    access_token=_git_hub_token,
    github_api_url="https://api.github.com",
    file_filter=lambda file_path: file_path.endswith(
        ".md"
    ),  # load all markdowns files.
)
documents = loader.load()

In [6]:
documents[1].page_content

'---------------------------\nCloudera QuickStart VM 6.3.2\nCentOS 7 + GNOME Based\nJava Eclipse & Scala Eclipse IDE Included\nMySql With \'retail_db\' Installed\n\n\n---------------------------\nMinimum System Requirement - 2/4 Cores + 16GB RAM\nhttps://www.peazip.org/\nhttp://bit.ly/GetVMPlayer\n\n---------------------------\nDownload In 3 Parts (Direct Links)\nPart 1 - https://bit.ly/CDH_6_3_2_CentOS7_1\nPart 2 - https://bit.ly/CDH_6_3_2_CentOS7_2\nPart 3 - https://bit.ly/CDH_6_3_2_CentOS7_3\nOR \nhttp://bit.ly/Minus1By12Repo\n\n\n---------------------------\nCentOS GUI Login \'Base User\' Password - BaseUser@123\n\'root\'  Password - BaseUser@123\n\n#\n#---------------------------\n#\nsudo user - osboxes\nsudo password - BaseUser@123\n\n#\n#---------------------------\n#\nMySql user - root\nMySql password - bigdata\n---------------------------\nCloudera Manager user - admin\nCloudera Manager password - admin\n---------------------------\nhttp://www.Minus1By12.com  \nhttp://bit.ly/M

In [ ]:
from crewai_tools import GithubSearchTool

# Initialize the tool with your PAT
github_tool = GithubSearchTool(
    github_repo='https://github.com/brijeshdhaker/docker-hadoop-cluster',
    gh_token=_git_hub_token,
    content_types=['code', 'issue']
)

# Use the tool within an Agent
from crewai import Agent

github_specialist = Agent(
    role='GitHub Researcher',
    goal='Search for specific code patterns and issues in the repository',
    backstory='Expert in navigating complex codebases and tracking development issues.',
    tools=[github_tool],
    verbose=True
)


